# 🔄 SUR: Cross-Price Elasticity Matrix
---

## Why SUR?

OLS estimates **own-price** elasticity only. But products compete:

> *"When Polyethylene price rises, some customers switch to Polypropylene"*

**Seemingly Unrelated Regressions (SUR)** estimates a system of demand equations jointly.

## System Specification

$$\begin{aligned}
\ln(Q_1) &= \alpha_1 + \varepsilon_{11} \ln(P_1) + \varepsilon_{12} \ln(P_2) + \cdots + u_1 \\
\ln(Q_2) &= \alpha_2 + \varepsilon_{21} \ln(P_1) + \varepsilon_{22} \ln(P_2) + \cdots + u_2 \\
&\vdots
\end{aligned}$$

## The Elasticity Matrix

$$\mathbf{E} = \begin{bmatrix} \varepsilon_{11} & \varepsilon_{12} & \cdots \\ \varepsilon_{21} & \varepsilon_{22} & \cdots \\ \vdots & \vdots & \ddots \end{bmatrix}$$

- **Diagonal:** Own-price elasticity (negative)
- **Off-diagonal positive:** Substitutes
- **Off-diagonal negative:** Complements

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session
session = get_active_session()

---
## 1. Prepare Family-Level Data

In [ ]:
fam_df = session.sql('''
    SELECT order_date, product_family, 
           SUM(total_quantity_mt) as quantity, AVG(avg_price_per_mt) as price
    FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA
    WHERE ln_quantity IS NOT NULL
    GROUP BY order_date, product_family
''').to_pandas()

families = sorted(fam_df['PRODUCT_FAMILY'].unique().tolist())
fam_ids = [f.replace(' ','_').upper() for f in families]
n = len(families)

print(f'📊 Product Families: {families}')
print(f'📊 Observations: {len(fam_df)}')

# Pivot to wide format
price_wide = fam_df.pivot(index='ORDER_DATE', columns='PRODUCT_FAMILY', values='price').reset_index()
qty_wide = fam_df.pivot(index='ORDER_DATE', columns='PRODUCT_FAMILY', values='quantity').reset_index()
sur = price_wide.merge(qty_wide, on='ORDER_DATE', suffixes=('_P','_Q'))

for f, fid in zip(families, fam_ids):
    sur[f'LN_P_{fid}'] = np.log(sur[f'{f}_P'].clip(lower=1))
    sur[f'LN_Q_{fid}'] = np.log(sur[f'{f}_Q'].clip(lower=1))

sur = sur.dropna()
print(f'📊 Clean observations: {len(sur)}')

---
## 2. Estimate SUR System

In [ ]:
print('🔄 ESTIMATING SUR SYSTEM')
print('=' * 70)

pcols = [f'LN_P_{f}' for f in fam_ids]
models = {}

for fid in fam_ids:
    y = sur[f'LN_Q_{fid}']
    X = sm.add_constant(sur[pcols])
    model = OLS(y, X).fit()
    models[fid] = model
    
    print(f'\n📊 {fid}: R²={model.rsquared:.4f}, F={model.fvalue:.2f}')
    for pc in pcols:
        pfid = pc.replace('LN_P_','')
        coef = model.params[pc]
        pval = model.pvalues[pc]
        sig = '***' if pval<0.01 else '**' if pval<0.05 else '*' if pval<0.1 else ''
        own = '← OWN' if pfid==fid else ''
        print(f'   {pfid:15s}: {coef:+8.4f} {sig:3s} {own}')

---
## 3. Build Elasticity Matrix

In [ ]:
E = pd.DataFrame(index=fam_ids, columns=fam_ids, dtype=float)
P = pd.DataFrame(index=fam_ids, columns=fam_ids, dtype=float)

for d in fam_ids:
    for p in fam_ids:
        E.loc[d,p] = models[d].params[f'LN_P_{p}']
        P.loc[d,p] = models[d].pvalues[f'LN_P_{p}']

print('\n📊 ELASTICITY MATRIX E[i,j] = ∂ln(Qᵢ)/∂ln(Pⱼ)')
print(E.round(4))

In [ ]:
# Visualization
fig = go.Figure(go.Heatmap(
    z=E.values.astype(float), x=E.columns, y=E.index,
    colorscale='RdBu', zmid=0, zmin=-2.5, zmax=1,
    text=E.round(3).values, texttemplate='%{text}',
    textfont={'size':14,'color':'white'},
    colorbar=dict(title='ε')
))
fig.update_layout(
    title='<b>Cross-Price Elasticity Matrix</b><br><sup>Row: Demand | Col: Price | Diagonal: Own-Price</sup>',
    xaxis_title='Price of Product j', yaxis_title='Demand for Product i',
    template='plotly_dark', height=500, width=600
)
fig.show()

---
## 4. Substitution Analysis

In [ ]:
print('🔍 SUBSTITUTION PATTERNS')
print('=' * 60)

print('\n📊 OWN-PRICE ELASTICITIES')
for i, f in enumerate(fam_ids):
    e = E.iloc[i,i]
    status = 'ELASTIC' if e < -1 else 'INELASTIC'
    print(f'   {f:15s}: ε = {e:+.4f} → {status}')

print('\n📊 SIGNIFICANT SUBSTITUTES (εᵢⱼ > 0, p < 0.05)')
for i, d in enumerate(fam_ids):
    for j, p in enumerate(fam_ids):
        if i != j and E.iloc[i,j] > 0 and P.iloc[i,j] < 0.05:
            print(f'   {d} ← {p}: ε = {E.iloc[i,j]:+.4f}')
            print(f'      → If {p} ↑10%, {d} demand ↑{E.iloc[i,j]*10:.1f}%')

---
## ✅ SUR Estimation Complete

**Proceed to:** `04_optimal_pricing` for portfolio optimization